# 🔬 Retinal OCT-C8 — Disease Classification
**Model:** EfficientNet-B3 fine-tuned  
**Dataset:** [retinal-oct-c8](https://www.kaggle.com/datasets/obulisainaren/retinal-oct-c8) — 8 classes, ~24 000 images  
**Classes:** AMD · CNV · CSR · DME · DR · Drusen · MH · Normal

## 1 · Setup & Imports

In [ ]:
import os, json, copy, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B3_Weights
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2 · Dataset Path

In [ ]:
# Kaggle mounts datasets under /kaggle/input/
DATASET_ROOT = Path('/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset')

# Verify structure
for split in ['train', 'val', 'test']:
    split_path = DATASET_ROOT / split
    if split_path.exists():
        classes = sorted(os.listdir(split_path))
        total = sum(len(os.listdir(split_path / c)) for c in classes)
        print(f'{split:5s}  {total:6,} images   classes: {classes}')
    else:
        print(f'{split:5s}  ✗ NOT FOUND at {split_path}')

## 3 · Config

In [ ]:
CFG = dict(
    img_size      = 224,
    batch_size    = 64,       # T4/P100 GPU handles 64 fine
    epochs        = 30,
    lr            = 1e-4,
    weight_decay  = 1e-4,
    freeze_epochs = 5,        # freeze backbone for first N epochs
    patience      = 7,        # early stopping
    num_workers   = 4,
    label_smooth  = 0.1,
    mean          = [0.485, 0.456, 0.406],
    std           = [0.229, 0.224, 0.225],
    output_dir    = Path('/kaggle/working/checkpoints'),
)
CFG['output_dir'].mkdir(exist_ok=True)
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

## 4 · Data Loaders

In [ ]:
S = CFG['img_size']

train_tf = transforms.Compose([
    transforms.Resize((S + 32, S + 32)),
    transforms.RandomCrop(S),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(CFG['mean'], CFG['std']),
])

val_tf = transforms.Compose([
    transforms.Resize((S, S)),
    transforms.ToTensor(),
    transforms.Normalize(CFG['mean'], CFG['std']),
])

train_ds = datasets.ImageFolder(DATASET_ROOT / 'train', transform=train_tf)
val_ds   = datasets.ImageFolder(DATASET_ROOT / 'val',   transform=val_tf)
test_ds  = datasets.ImageFolder(DATASET_ROOT / 'test',  transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

### Preview training samples

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.patch.set_facecolor('#0D1117')
for ax, cls in zip(axes.flat, CLASS_NAMES):
    cls_dir = DATASET_ROOT / 'train' / cls
    img_path = next(cls_dir.iterdir())
    from PIL import Image
    img = Image.open(img_path).convert('RGB').resize((224, 224))
    ax.imshow(img)
    ax.set_title(cls, color='white', fontsize=11, fontweight='bold')
    ax.axis('off')
plt.suptitle('One sample per class', color='#58A6FF', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5 · Model — EfficientNet-B3

In [ ]:
def build_model(num_classes):
    m = models.efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
    in_feat = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(0.3, inplace=True),
        nn.Linear(in_feat, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(512, num_classes),
    )
    return m

def freeze_backbone(m):
    for name, p in m.named_parameters():
        if 'classifier' not in name: p.requires_grad = False

def unfreeze_backbone(m):
    for p in m.parameters(): p.requires_grad = True

model = build_model(NUM_CLASSES).to(device)

total  = sum(p.numel() for p in model.parameters())
print(f'Total params  : {total:,}')
print(f'Model ready on {device}')

## 6 · Training

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smooth'])
optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)

def train_epoch(model, loader):
    model.train()
    loss_sum, correct, total = 0., 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    loss_sum, correct, total = 0., 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        loss_sum += criterion(out, labels).item() * imgs.size(0)
        preds = out.argmax(1)
        correct  += (preds == labels).sum().item()
        total    += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return loss_sum / total, correct / total, all_preds, all_labels

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc   = 0.
best_val_loss  = float('inf')
best_weights   = None
patience_cnt   = 0

print(f"{'Epoch':>6}  {'T-Loss':>8}  {'T-Acc':>7}  {'V-Loss':>8}  {'V-Acc':>7}  {'Time':>6}")
print('-' * 56)

for epoch in range(1, CFG['epochs'] + 1):
    t0 = time.time()

    # Phase control
    if epoch == 1:
        freeze_backbone(model)
    if epoch == CFG['freeze_epochs'] + 1:
        unfreeze_backbone(model)
        for pg in optimizer.param_groups:
            pg['lr'] = CFG['lr'] * 0.1
        print('  ↳ Backbone unfrozen — full fine-tune mode')

    tr_loss, tr_acc = train_epoch(model, train_loader)
    vl_loss, vl_acc, _, _ = eval_epoch(model, val_loader)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    elapsed = time.time() - t0
    flag = ''
    if vl_acc > best_val_acc:
        best_val_acc  = vl_acc
        best_weights  = copy.deepcopy(model.state_dict())
        torch.save(best_weights, CFG['output_dir'] / 'best_model.pth')
        flag = '  ✓ saved'

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_cnt  = 0
    else:
        patience_cnt += 1

    print(f"{epoch:6d}  {tr_loss:8.4f}  {tr_acc:7.4f}  {vl_loss:8.4f}  {vl_acc:7.4f}  {elapsed:5.1f}s{flag}")

    if patience_cnt >= CFG['patience']:
        print(f'\nEarly stopping at epoch {epoch}')
        break

print(f'\nBest Val Accuracy: {best_val_acc:.4f}')

## 7 · Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0D1117')
for ax in (ax1, ax2):
    ax.set_facecolor('#161B22')
    for spine in ax.spines.values(): spine.set_color('#30363D')
    ax.tick_params(colors='#8B949E')
    ax.xaxis.label.set_color('#8B949E')
    ax.yaxis.label.set_color('#8B949E')

ax1.plot(epochs_ran, history['train_loss'], color='#58A6FF', lw=2, label='Train')
ax1.plot(epochs_ran, history['val_loss'],   color='#FF6B6B', lw=2, label='Val',   linestyle='--')
ax1.set_title('Loss', color='#E6EDF3', fontweight='bold')
ax1.legend(facecolor='#21262D', labelcolor='white')

ax2.plot(epochs_ran, history['train_acc'], color='#58A6FF', lw=2, label='Train')
ax2.plot(epochs_ran, history['val_acc'],   color='#3FB950', lw=2, label='Val',   linestyle='--')
ax2.set_title('Accuracy', color='#E6EDF3', fontweight='bold')
ax2.legend(facecolor='#21262D', labelcolor='white')

plt.suptitle('Training History', color='#58A6FF', fontsize=14)
plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'training_curves.png', dpi=150, bbox_inches='tight',
            facecolor='#0D1117')
plt.show()

## 8 · Test Evaluation

In [ ]:
model.load_state_dict(best_weights)
test_loss, test_acc, preds, labels = eval_epoch(model, test_loader)

print(f'Test Loss : {test_loss:.4f}')
print(f'Test Acc  : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print()
print(classification_report(labels, preds, target_names=CLASS_NAMES))

## 9 · Confusion Matrix

In [ ]:
cm = confusion_matrix(labels, preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0D1117')
ax.set_facecolor('#0D1117')

sns.heatmap(cm_norm, annot=True, fmt='.2f',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cmap='Blues', linewidths=0.5, linecolor='#21262D',
            ax=ax, cbar_kws={'shrink': 0.8})

ax.set_xlabel('Predicted', color='#8B949E', fontsize=11)
ax.set_ylabel('True',      color='#8B949E', fontsize=11)
ax.set_title('Normalised Confusion Matrix', color='#58A6FF', fontsize=14, pad=14)
ax.tick_params(colors='#C9D1D9')

plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'confusion_matrix.png', dpi=150, bbox_inches='tight',
            facecolor='#0D1117')
plt.show()

## 10 · Save Artifacts

In [ ]:
# class_names.json  (needed by app.py)
class_map = {str(i): name for i, name in enumerate(CLASS_NAMES)}
with open(CFG['output_dir'] / 'class_names.json', 'w') as f:
    json.dump(class_map, f, indent=2)

# training history
with open(CFG['output_dir'] / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('Saved to', CFG['output_dir'])
for p in sorted(CFG['output_dir'].iterdir()):
    size = p.stat().st_size
    print(f'  {p.name:<30}  {size/1e6:.2f} MB' if size > 1e5 else f'  {p.name}')

---
## ✅ Done!
Download `checkpoints/best_model.pth` and `checkpoints/class_names.json`  
then run `app.py` locally:
```bash
python app.py --model best_model.pth --classes class_names.json
```